# 05 Select

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Choose, compute and rename columns with <code>select</code>, <code>selectExpr</code> and column expressions, and know the different ways to refer to a column.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
<code>select</code> is the most used transformation in Spark. Picking only the columns you need also makes queries faster, because Spark reads less data from columnar formats like Parquet and Delta (column pruning). It is also the first place where people meet the confusing "string vs <code>col()</code> vs <code>df.col</code>" question.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What does select do?</b><br>
<code>select</code> returns a <b>new DataFrame</b> containing only the columns (or expressions) you list.
<ul><li>A column can be referenced as a string <code>"salary"</code>, as <code>F.col("salary")</code>, as <code>df.salary</code> or as <code>df["salary"]</code></li><li>A <b>column expression</b> is a formula built from columns, such as <code>F.col("salary") * 1.1</code></li><li><code>selectExpr</code> takes SQL snippets as strings, such as <code>"salary * 1.1 AS new_salary"</code></li><li><code>select</code> is a <b>narrow transformation</b>: each partition is processed on its own, no shuffle</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A Bronze table of web clickstream events has 140 columns. The marketing Gold table needs 6 of them plus a computed <code>session_minutes</code> column. One <code>select</code> picks the 6, computes the seventh, and because Delta is columnar, Spark never reads the other 134 columns from storage.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df.select("name", "salary")                           # strings
df.select(F.col("name"), F.col("salary") * 1.1)       # column expressions
df.select(df.name, df["salary"])                      # attribute / item access
df.select("*", (F.col("salary") * 0.1).alias("bonus"))  # all columns plus a new one
df.selectExpr("name", "salary * 1.1 AS new_salary")   # SQL snippets
```

| Way to reference a column | Example | Notes |
|---|---|---|
| String | `"salary"` | Simplest. Can't do maths on it directly |
| `F.col()` | `F.col("salary")` | Most flexible. Works before the DataFrame variable exists |
| Attribute | `df.salary` | Short, but breaks for names with spaces or that clash with methods (`df.count`) |
| Item | `df["salary"]` | Works with any name. Ties the column to that specific DataFrame |

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b><br>
Prefer <code>F.col("name")</code> in pipelines. It is not tied to a specific DataFrame variable, so it keeps working when you chain transformations.
</div>

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an employee DataFrame used by every example in this notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 6-row table with columns <code>emp_id</code>, <code>first_name</code>, <code>last_name</code>, <code>dept</code>, <code>salary</code>, <code>bonus_pct</code> and <code>city</code>.
</div>

In [0]:
from pyspark.sql import functions as F

employees = spark.createDataFrame(
    [
        (101, "Amal", "Raj", "Data", 21000, 0.10, "Chennai"),
        (102, "Senthil", "Kumar", "Data", 150000, 0.15, "New York"),
        (103, "John", "Smith", "Finance", 18000, 0.05, "London"),
        (104, "Vivek", "Nair", "Sales", 25000, 0.20, "Dubai"),
        (105, "Priya", "Iyer", "Finance", 42000, None, "Chennai"),
        (106, "Sara", "Ali", "Sales", 39000, 0.12, "Dubai"),
    ],
    "emp_id INT, first_name STRING, last_name STRING, dept STRING, salary INT, bonus_pct DOUBLE, city STRING",
)
display(employees)

## 1. Select columns by name

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Selects a subset of columns using strings, and also passes a Python list of names.<br><br>
<b>Why it matters</b><br>Passing a list lets you build the column list in code, for example from a config file.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two tables, each with only the requested columns and all 6 rows.
</div>

In [0]:
# Pass column names as separate arguments.
employees.select("first_name", "dept", "salary").show()

# Or build the list in Python and pass it. Useful when columns come from config.
wanted = ["emp_id", "city"]
employees.select(wanted).show()

## 2. Four ways to reference a column

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Selects the same column four different ways and proves the results are the same.<br><br>
<b>Why it matters</b><br>You will see all four styles in other people's code. You need to read them all, and choose consistently in your own.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Four identical single-column results, then <code>True</code>.
</div>

In [0]:
a = employees.select("salary")
b = employees.select(F.col("salary"))
c = employees.select(employees.salary)
d = employees.select(employees["salary"])

# All four produce the same data.
print(a.collect() == b.collect() == c.collect() == d.collect())

# Why df.attribute can surprise you: 'count' is a DataFrame METHOD, not a column.
tricky = spark.createDataFrame([(1, 5)], "id INT, count INT")
print(type(tricky.count))          # a bound method, not a Column
tricky.select(tricky["count"]).show()   # item access works for any name

## 3. Compute new columns with expressions and alias

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds calculated columns (annual bonus, full name) inside <code>select</code> and names them with <code>alias</code>.<br><br>
<b>Why it matters</b><br>Real pipelines compute columns all the time. Without <code>alias</code>, Spark generates ugly names like <code>(salary * bonus_pct)</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table with <code>full_name</code>, <code>salary</code>, <code>bonus_amount</code> and <code>salary_in_k</code>. Priya's bonus is <code>null</code>, because <code>bonus_pct</code> is null.
</div>

In [0]:
employees.select(
    F.concat_ws(" ", "first_name", "last_name").alias("full_name"),
    "salary",
    (F.col("salary") * F.col("bonus_pct")).alias("bonus_amount"),
    F.round(F.col("salary") / 1000, 1).alias("salary_in_k"),
).show()

# Without alias, the column name is the expression text:
employees.select(F.col("salary") * 2).printSchema()

## 4. Keep everything and add more: `"*"`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Selects every existing column with <code>"*"</code>, then adds a literal and a computed column.<br><br>
<b>Why it matters</b><br><code>select("*", ...)</code> is a common way to add several columns at once. <code>F.lit()</code> creates a constant column, which you often need for audit fields such as a load timestamp or a source system name.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
All 7 original columns plus <code>source_system</code> (always <code>HR</code>) and <code>loaded_at</code> (the current timestamp).
</div>

In [0]:
employees.select(
    "*",
    F.lit("HR").alias("source_system"),
    F.current_timestamp().alias("loaded_at"),
).show(truncate=False)

## 5. selectExpr: SQL inside the DataFrame API

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same kind of logic as SQL strings with <code>selectExpr</code>.<br><br>
<b>Why it matters</b><br>Handy for SQL-fluent teams and for logic stored as strings in config. <code>F.expr()</code> lets you mix a SQL snippet into a normal <code>select</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table with <code>first_name</code>, <code>salary_eur</code> and <code>band</code>, then a table using <code>F.expr</code> with the same <code>band</code> logic.
</div>

In [0]:
employees.selectExpr(
    "first_name",
    "round(salary * 0.92, 2) AS salary_eur",
    "CASE WHEN salary >= 40000 THEN 'senior' ELSE 'junior' END AS band",
).show()

# F.expr() lets you use one SQL snippet inside a regular select.
employees.select(
    "first_name",
    F.expr("CASE WHEN salary >= 40000 THEN 'senior' ELSE 'junior' END").alias("band"),
).show()

## 6. Select columns dynamically

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Picks columns by pattern, by type, and drops some from the list, all with plain Python over <code>df.columns</code> and <code>df.dtypes</code>.<br><br>
<b>Why it matters</b><br>Generic pipeline code (for example "cast every <code>_amount</code> column to decimal") is built this way. <code>colRegex</code> selects by regular expression.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Columns ending in <code>_name</code>, then all numeric columns, then every column except <code>city</code>, then <code>first_name</code> and <code>last_name</code> from <code>colRegex</code>.
</div>

In [0]:
# Columns whose name ends with "_name"
name_cols = [c for c in employees.columns if c.endswith("_name")]
employees.select(name_cols).show(2)

# Columns of numeric types, found from dtypes (a list of (name, type) pairs)
numeric_cols = [name for name, dtype in employees.dtypes if dtype in ("int", "bigint", "double")]
employees.select(numeric_cols).show(2)

# Every column except one
employees.select([c for c in employees.columns if c != "city"]).show(2)

# colRegex takes a regex wrapped in backticks
employees.select(employees.colRegex("`.*_name`")).show(2)

## 7. Rename many columns at once

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses a list comprehension inside <code>select</code> to rename every column, for example to upper case or with a prefix.<br><br>
<b>Why it matters</b><br>Standardising column names (snake_case, prefixes per source) is routine in the Silver layer. Doing it in one <code>select</code> builds a small plan, unlike calling <code>withColumnRenamed</code> in a loop.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same 6 rows with columns named <code>HR_EMP_ID</code>, <code>HR_FIRST_NAME</code> and so on.
</div>

In [0]:
renamed = employees.select([F.col(c).alias(f"hr_{c}".upper()) for c in employees.columns])
renamed.show(2)
print(renamed.columns)

## 8. Edge cases: spaces, dots and missing columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Handles column names with spaces or dots, and shows the error when a column doesn't exist.<br><br>
<b>Why it matters</b><br>Files from Excel or business systems often have names like <code>Order Date</code> or <code>amount.usd</code>. A dot normally means "field inside a struct", so these names need <b>backticks</b>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The awkward columns are selected correctly using backticks. Selecting a missing column prints an <code>UNRESOLVED_COLUMN</code> (AnalysisException) message that suggests similar names.
</div>

In [0]:
awkward = spark.createDataFrame([(1, "2024-01-01", 9.5)], ["id", "Order Date", "amount.usd"])

# Backticks tell Spark the dot is part of the name, not a struct field.
awkward.select(F.col("`Order Date`"), F.col("`amount.usd`")).show()

# selectExpr needs backticks too.
awkward.selectExpr("`amount.usd` * 2 AS doubled").show()

# A missing column fails at analysis time, before any data is read.
try:
    employees.select("salaryy").show()
except Exception as e:
    print(type(e).__name__, "-", str(e).split("\n")[0][:160])

## Under the hood

```
employees.select("first_name", (col("salary") * 1.1).alias("raise"))
        |
        v
 Logical plan:  Project [first_name, (salary * 1.1) AS raise]
        |
        v
 Optimizer:     column pruning -> only first_name and salary are read
        |
        v
 Physical plan: Project runs per partition (narrow, no shuffle)
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `select` only adds a **Project** node to the plan.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> every partition is processed independently, so no data moves between executors.

Several `select` and `withColumn` calls in a row are **collapsed into one Project** by the optimizer (rule: CollapseProject).

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Small-data gotcha</b><br>
On a tiny DataFrame built from a Python list, the optimizer simply pre-computes the result and the plan shows <code>LocalTableScan</code>, hiding the interesting parts. To study plans, generate data with <code>spark.range()</code> as below.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <b>Parsed</b> and <b>Analyzed</b> plans show two <code>Project</code> nodes (one per select). The <b>Optimized</b> and <b>Physical</b> plans show a single <code>Project</code> over a <code>Range</code> scan.
</div>

In [0]:
plan_df = (
    spark.range(1_000_000)                                   # a real (non-local) source
    .select("id", (F.col("id") % 50_000).alias("salary"), F.lit("Data").alias("dept"))
    .select("id", (F.col("salary") * 1.1).alias("raise"))    # second select
)
plan_df.explain(True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>TypeError</code> when doing maths on a string name</b><br>
<code>df.select("salary" * 2)</code> multiplies the Python string. Use <code>F.col("salary") * 2</code>.<br><br>
**⛔ Problem: <code>[UNRESOLVED_COLUMN]</code> error**<br>Typo or wrong case in a column name, or the column was dropped earlier in the chain. Check <code>df.columns</code>. The error message suggests close matches.<br><br>
**⛔ Problem: <code>df.count</code> or <code>df.first</code> returns a method, not a column**<br>The column name clashes with a DataFrame method. Use <code>df["count"]</code> or <code>F.col("count")</code>.<br><br>
<b>⛔ Problem: column names with dots return nulls or errors</b><br>Spark treats the dot as a struct field. Wrap the name in backticks: <code>F.col("`amount.usd`")</code>.<br><br>
<b>⛔ Problem: ambiguous column after a join</b><br><code>df1.id</code> and <code>df2.id</code> both exist. Use DataFrame aliases (lesson 17).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What does <code>select</code> do and is it a transformation or an action?</b><br>
It returns a new DataFrame with the chosen columns or expressions. It is a narrow transformation, so it is lazy and needs no shuffle.<br><br>

<b>🎤 2. What is the difference between <code>select</code> and <code>selectExpr</code>?</b><br>
<code>select</code> takes column names or Column objects. <code>selectExpr</code> takes SQL expression strings, such as <code>"salary * 1.1 AS raise"</code>. They compile to the same plan.<br><br>

<b>🎤 3. What are the ways to reference a column, and which do you prefer?</b><br>
A string, <code>F.col("x")</code>, <code>df.x</code> and <code>df["x"]</code>. I prefer <code>F.col</code> because it isn't bound to a DataFrame variable and works for any name. <code>df["x"]</code> is useful to disambiguate after joins.<br><br>

<b>🎤 4. Why does selecting fewer columns make a query faster?</b><br>
Delta and Parquet are columnar. The optimizer pushes the column list down to the scan (column pruning), so Spark reads only those columns from storage and moves less data.<br><br>

<b>🎤 5. What is the difference between <code>select</code> and <code>withColumn</code>?</b><br>
<code>withColumn</code> adds or replaces one column and keeps the rest. <code>select</code> defines the full output column list. To add many columns, one <code>select</code> is cleaner than many <code>withColumn</code> calls.<br><br>

<b>🎤 6. How do you select a column whose name contains a dot or a space?</b><br>
Wrap it in backticks, for example <code>F.col("`order.date`")</code>. Without backticks a dot is read as struct field access.<br><br>

<b>🎤 7. What is <code>F.lit()</code> used for?</b><br>
To create a column with a constant value, for example a source system name or a default flag.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which code returns a DataFrame with <code>name</code> and a new column <code>tax</code> equal to 20% of <code>price</code>?</b><br>
A. <code>df.select("name", "price" * 0.2)</code><br>
B. <code>df.select("name", (F.col("price") * 0.2).alias("tax"))</code><br>
C. <code>df.select("name").withColumn("tax", F.col("price") * 0.2)</code><br>
D. <code>df.selectExpr("name", "tax = price * 0.2")</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> A multiplies a Python string. C fails because <code>price</code> is no longer in the DataFrame after the first select. D is not valid SQL aliasing (use <code>price * 0.2 AS tax</code>).</details><br><br>

<b>Q2. A Delta table has 200 columns. A query selects 3 of them. What does Spark read from storage?</b><br>
A. All 200 columns, then discards 197<br>
B. Only the 3 selected columns, thanks to column pruning<br>
C. Only the first 3 columns of each file<br>
D. Only the rows that match the 3 columns<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Columnar formats let Spark read only the needed columns.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using the <code>employees</code> DataFrame:
<ol><li>Select <code>emp_id</code> and <code>city</code> in three different column-reference styles</li><li>Return <code>first_name</code>, <code>salary</code> and a column <code>monthly_salary</code> (salary / 12, rounded to 2 decimals)</li><li>With <code>selectExpr</code>, return <code>first_name</code> and <code>salary_band</code>: <code>'high'</code> if salary > 30000 else <code>'low'</code></li><li>Return every column plus <code>country</code> set to the constant <code>'unknown'</code></li><li>Select only the columns of type <code>string</code>, found dynamically from <code>dtypes</code></li><li>Rename all columns to upper case in one <code>select</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Three reference styles
employees.select("emp_id", "city").show(2)
employees.select(F.col("emp_id"), F.col("city")).show(2)
employees.select(employees["emp_id"], employees.city).show(2)

# 2. Computed column with rounding
employees.select("first_name", "salary", F.round(F.col("salary") / 12, 2).alias("monthly_salary")).show()

# 3. SQL expression
employees.selectExpr("first_name", "CASE WHEN salary > 30000 THEN 'high' ELSE 'low' END AS salary_band").show()

# 4. All columns plus a constant
employees.select("*", F.lit("unknown").alias("country")).show(2)

# 5. String columns only
string_cols = [n for n, t in employees.dtypes if t == "string"]
employees.select(string_cols).show(2)

# 6. Upper-case names in one select
employees.select([F.col(c).alias(c.upper()) for c in employees.columns]).show(2)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>select</code> returns a new DataFrame with the columns or expressions you list</li><li>It is a lazy, narrow transformation: a <code>Project</code> node in the plan, no shuffle</li><li>Reference columns with <code>"x"</code>, <code>F.col("x")</code>, <code>df.x</code> or <code>df["x"]</code>. Prefer <code>F.col</code></li><li>Use <code>alias</code> to name computed columns, and <code>F.lit</code> for constants</li><li><code>selectExpr</code> and <code>F.expr</code> accept SQL snippets</li><li>Build column lists in Python from <code>df.columns</code> and <code>df.dtypes</code> for generic code</li><li>Backticks for names with dots or spaces</li><li>Selecting fewer columns reads less data from columnar formats (column pruning)</li></ul>
</div>

| Task | Code |
|---|---|
| Pick columns | `df.select("a", "b")` |
| Compute | `df.select((F.col("a") * 2).alias("a2"))` |
| All plus new | `df.select("*", F.lit(1).alias("one"))` |
| SQL snippet | `df.selectExpr("a * 2 AS a2")` |
| Rename all | `df.select([F.col(c).alias(c.lower()) for c in df.columns])` |
| Regex | ``df.select(df.colRegex("`^amt_.*`"))`` |

## Git commit

```
git add 01_spark_basics/05_select.ipynb
git commit -m "add 05 select notebook"
```